In [309]:
import pandas as pd
import re, unicodedata

In [310]:
path_to_candidate = "final_candidates/AI_bert_2021_seed_4838_strict.csv"
path_to_labels = "labelled_news/AI_bert_2021_seed_4838_strict_topics.xlsx"
path_to_new_csv = "labelled_subs_2021.csv"

In [311]:
df = pd.read_csv(path_to_candidate, index_col=0)
# df.head()

In [312]:
# #inspect topic probabilities
# print("Min probability :", df['probability'].min())
# print("Max probability :", df['probability'].max())
# print("Mean probability:", df['probability'].mean())
# print("Std deviation   :", df['probability'].std())


In [313]:
def norm_topic(x):
    """Canonicalise topic ids for safe matching across df and Excel."""
    if pd.isna(x):
        return None
    s = str(x)
    # unify unicode + strip invisible/space noise
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", "", s)

    # if it looks numeric, coerce to a canonical string:
    #  - 10.0 -> "10"
    #  - 10.50 -> "10.5"
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        else:
            # normalise trimming trailing zeros/dot
            s2 = ("%.15g" % f)  # compact float repr without scientific if possible
            return s2
    except ValueError:
        # non-numeric topic ids like 'FD50' are kept as-is (case-sensitive)
        return s.strip()

# --- 1) Load mapping and normalise its index ---
mapping = pd.read_excel(path_to_labels, index_col=0)
mapping.index = mapping.index.map(norm_topic)

# Optionally: keep only rows that actually have a label
mapping_labeled = mapping[~mapping["Label NL"].isna()].copy()

# --- 2) Normalise df topics too ---
df["topic_norm"] = df["topic"].map(norm_topic)

# --- 3) Build maps and assign ---
label_map_nl = mapping_labeled["Label NL"]
meta_map_nl  = mapping_labeled["Meta NL"]

label_map = mapping_labeled["Label"]
meta_map  = mapping_labeled["Meta"]

df["topic_label_nl"] = df["topic_norm"].map(label_map_nl)
df["topic_meta_nl"]  = df["topic_norm"].map(meta_map_nl)

df["topic_label"] = df["topic_norm"].map(label_map)
df["topic_meta"]  = df["topic_norm"].map(meta_map)

# --- 4) (Optional) sanity checks ---
# What fraction matched?
matched_frac = df["topic_label"].notna().mean()
print(f"Matched labels for {matched_frac:.1%} of rows.")

# If you want to see why some didn’t match, compare key sets:
missing_keys = set(df["topic_norm"].dropna().unique()) - set(mapping_labeled.index)
if missing_keys:
    print(f"{len(missing_keys)} df topic ids not found in mapping (showing up to 20):",
          sorted(list(missing_keys))[:20])

# --- 5) Keep only labelled rows (discard unlabelled) ---
df_labeled = df[df["topic_label"].notna()].copy()
# If you’re done with the helper column:
# df_labeled.drop(columns=["topic_norm"], inplace=True)


Matched labels for 100.0% of rows.


In [314]:
# # show rows where topic_label is 'NOISE'
# noise_rows = df_labeled[df_labeled['topic_label'] == 'NOISE']
# print(f"Rows labeled as 'NOISE': {len(noise_rows)}")

# # show row numbers of rows labeled as 'NOISE'
# print("Row numbers of 'NOISE' rows:", noise_rows.index.tolist())

# # print one row of 'NOISE' rows to inspect
# if not noise_rows.empty:
#     print("Example 'NOISE' row:")
#     print(noise_rows.iloc[0])
    


In [315]:
# df.head()

In [316]:
df = df.dropna(subset=['topic_label']).copy()
df = df[df['topic_meta'] != 'NOISE'].reset_index(drop=True)

print(f"rows dropped due to missing labels: {len(df) - len(df_labeled)}")




rows dropped due to missing labels: 0


In [317]:
df.head()

,title,outlet,date,authors,body,word_count,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,...,meta,label_nl,meta_nl,date_parsed,year,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta
0,Een mensenleven kun je niet vatten in data Een...,TR,2021-06-25,LODEWIJK DROS,Techniek interview\nIn de toeslagenaffaire gaa...,255,yes,[],"['ai', 'kunstmatige intelligentie']","['ai', 'kunstmatige intelligentie']",...,AI RISKS & ETHICS,AI-technologie & risico’s,AI-risico’s & ethiek,2021-06-25,2021,19,menselijke intelligentie,WETENSCHAP & GEZONDHEID,human intelligence,SCIENCE & HEALTH
1,Robotracisme: hoe AI leert te discrimineren Ro...,TR,2021-04-01,RUFUS KAIN,Het is een wrange scène: een jonge studente di...,193,yes,['ai'],['algoritme'],"['ai', 'algoritme']",...,AI FOR SECURITY,Gezichtsherkenning & rechtshandhaving,AI voor veiligheid,2021-04-01,2021,0,"kunst, gezichtsherkenning en deepfakes",AI RISICO'S & ETHIEK,"art, facial recognition and deepfakes",AI RISKS & ETHICS
2,Gebruik meer slimme techniek tegen corona Gebr...,TR,2021-04-01,BART CUSTERS ANNIE MEUWESE,Coronabestrijding\nNaarmate meer mensen gevacc...,232,yes,[],"['ai', 'algoritme', 'artificiële intelligentie']","['ai', 'algoritme', 'artificiële intelligentie']",...,AI RISKS & ETHICS,COVID-19-pandemie & desinformatie,AI-risico’s & ethiek,2021-04-01,2021,5,pandemieën en vaccins,WETENSCHAP & GEZONDHEID,pandemics and vaccins,SCIENCE & HEALTH
3,De overheid is onvoldoende kritisch op algorit...,TR,2021-01-27,"STEFAN KEUKENKAMP, REDACTIE BINNENLAND",Bij het gebruik van voorspellende algoritmes d...,228,yes,['algoritmes'],['algoritmes'],['algoritmes'],...,EDUCATION,Universitaire opleiding & toekomstige technologie,Onderwijs,2021-01-27,2021,2,Nederlands kinderopvangschandaal,AI RISICO'S & ETHIEK,dutch childcare schandal,AI RISKS & ETHICS
4,Met zelfvarend schip kan stuurman aan wal blij...,TR,2021-08-28,HANS NAUTA,Je ziet nog maar weinig mensen in de Rotterdam...,262,yes,[],"['ai', 'algoritme']","['ai', 'algoritme']",...,ENVIRONMENT,Energiebronnen,Milieu,2021-08-28,2021,21,geld en zaken,BEDRIJF & FINANCIËN,money and business,BUSINESS & FINANCE


In [318]:
# overwrite robots & educatie (3) to 1) robots adn 2) tech & ai ontwikkeling

In [319]:
# df['topic_label'].value_counts()

In [320]:
mask = (
    (df["topic_label"] == "robots en educatie") &
    (df["topic_meta"] == "EDUCATIE")
)

df.loc[mask, "topic_label"] = "robots"
df.loc[mask, "topic_meta"] = "TECH & AI ONTWIKKELING"

In [321]:
# df.shape

In [322]:
df.to_csv(path_to_new_csv)

In [ ]:
# combine all csv in final_df folder to one csv
import glob
all_files = glob.glob("final_df/*.csv")
df_list = []
for filename in all_files:
    df_temp = pd.read_csv(filename, index_col=0)
    df_list.append(df_temp)
final_df = pd.concat(df_list, ignore_index=True)

 

In [334]:
final_df.shape

(13231, 26)

In [350]:
# change all instances of 'samenleving' to 'maatschappij'in topic_meta_nl
final_df['topic_meta_nl'] = final_df['topic_meta_nl'].replace('AI APPLICATIE IN INDUSTRIE', 'AI TOEPASSING IN INDUSTRIEËN')

In [353]:
final_df['topic_meta_nl'].value_counts()
# show number of topics
print(f"Number of unique topics in 'topic_meta_nl': {final_df['topic_meta'].nunique()}")

Number of unique topics in 'topic_meta_nl': 17


In [347]:
# show which row has topic_meta_nl as 'NUTRISCORE'
nutriscore_rows = final_df[final_df['topic_meta_nl'] == 'NUTRISCORE']
print(f"Rows with 'NUTRISCORE' in 'topic_meta_nl': {len(nutriscore_rows)}")
if not nutriscore_rows.empty:
    print("Example row with 'NUTRISCORE':")
    print(nutriscore_rows.iloc[0])

Rows with 'NUTRISCORE' in 'topic_meta_nl': 17
Example row with 'NUTRISCORE':
title                                   Warm havertaartje Warm havertaartje
outlet                                                                   TR
date                                                             2024-11-04
authors                                          JANINE EN ANNEMIEKE JANSEN
body                      Voor onze bijdrage van vorige week wierpen we ...
word_count                                                              270
ai_related                                                              yes
matched_keywords_title                                                   []
matched_keywords_body                                                ['ai']
matched_keywords_all                                                 ['ai']
n_hits_title_total                                                        0
n_hits_body_total                                                         3
company_hit

In [330]:
final_df.drop(columns=['label', 'meta', 'label_nl', 'meta_nl'], inplace=True)

In [335]:
final_df.tail()

,title,outlet,date,authors,body,word_count,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,...,verbs,date_parsed,year,topic,probability,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta
13226,De nieuwe versie van ChatGPT blundert als vano...,VK,2025-08-12,LAURENS VERHAGEN,1.\nOpenAI maakt de hype niet waar\nToen OpenA...,207,yes,['chatgpt'],"['ai', 'chatgpt', 'generatieve ai', 'gpt', 'op...","['ai', 'chatgpt', 'generatieve ai', 'gpt', 'op...",...,maken lanceren maken produceren opviel geven u...,2025-08-12,2025,19,0.055403,19,AI-modellen,TECH & AI ONTWIKKELING,AI models,TECH & AI DEVELOPMENT
13227,Een superbrein dat de mens voorbijstreeft? Een...,VK,2025-06-07,LAURENS VERHAGEN,Technologie 'Denkende' en 'begrijpende' AI\n'C...,297,yes,[],"['ai', 'anthropic', 'chatgpt', 'deepmind', 'll...","['ai', 'anthropic', 'chatgpt', 'deepmind', 'll...",...,begrijpen zien denken begrijpen lezen denken z...,2025-06-07,2025,16,0.037080,16,chatbots en desinformatie,AI RISICO'S & ETHIEK,chatbots and desinformation,AI RISKS & ETHICS
13228,Grote AI-denker bleef gefascineerd door de men...,VK,2025-08-22,LAURENS VERHAGEN,Margaret Boden (1936-2025)\nIs het geheel nieu...,255,yes,['ai'],"['ai', 'chatgpt', 'kunstmatige intelligentie']","['ai', 'chatgpt', 'kunstmatige intelligentie']",...,boden maken uitspuwen buitelen gebruiken overl...,2025-08-22,2025,15,0.008852,15,AI en tech trends,TECH & AI ONTWIKKELING,AI and tech trends,TECH & AI DEVELOPMENT
13229,De 'nerdy' miljardair achter DeepSeek De 'nerd...,VK,2025-01-29,FRANK RENSEN,Liang Wenfeng\nUit het niets verscheen de AI-a...,278,yes,[],"['ai', 'chatbot', 'chatgpt', 'openai']","['ai', 'chatbot', 'chatgpt', 'openai']",...,verschijnen overvleuglen heten vestigen gelden...,2025-01-29,2025,1,0.033187,1,chipindustrie,AI INFRASTRUCTUUR & CHIPINDUSTRIE,chip industry,AI INFRASTRUCTURE & CHIP INDUSTRY
13230,AI zoals wij die willen AI zoals wij die willen,VK,2025-08-30,LAURENS VERHAGEN,Kunstmatige intelligentie Specialist pleit voo...,234,yes,['ai'],"['ai', 'kunstmatige intelligentie', 'openai']","['ai', 'kunstmatige intelligentie', 'openai']",...,pleiten overnemen komen staan vrezen leunen vo...,2025-08-30,2025,31,0.110399,31,tech en overheid,POLITIEK & RECHT,tech and government,POLITICS & LAW


In [354]:
final_df.to_csv('final_df/combined_final_df.csv')